# Medical LLM Evaluation: Comparing Baseline vs. Fine-Tuned (QLoRA)

This notebook performs a comprehensive evaluation of the fine-tuned `Mistral-7B-Instruct-v0.2` model. We compare its performance against the baseline results generated in Notebook 02 using three distinct evaluation methodologies:

1.  **Lexical Overlap (ROUGE):** Measuring n-gram overlap between generated and ground-truth answers.
2.  **Semantic Similarity (BERTScore):** Using contextual embeddings to capture meaning beyond exact word matches.
3.  **LLM-as-Judge (Gemini 1.5 Flash):** Using a superior model to grade both versions on medical accuracy, professional tone, and conciseness.

**Runtime Requirement:** T4 GPU (for loading the 7B model and running inference).

### Cell 1 — Install Dependencies

We install the necessary libraries for model loading (`transformers`, `peft`, `bitsandbytes`), dataset handling (`datasets`), and evaluation metrics (`rouge-score`, `bert-score`). We also include `google-generativeai` to interact with Gemini for the LLM-as-Judge phase.

In [ ]:
!pip install -q transformers peft bitsandbytes accelerate datasets \
    rouge-score bert-score groq sentencepiece

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 6.0 MB/s eta 0:00:00


### Cell 2 — Environment Setup & Baseline Data Loading

We mount Google Drive to access our persistent project directory and retrieve the baseline results saved in Notebook 02. We also authenticate with HuggingFace Hub to pull the fine-tuned adapter weights.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')

import os
import json
import torch

PROJECT_DIR = '/content/drive/MyDrive/medical-llm'
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
HF_TOKEN = userdata.get('HF_TOKEN')

from huggingface_hub import login
login(token=HF_TOKEN)

print("Setup complete")
print(f"Project dir: {PROJECT_DIR}")

# Load baseline results
with open(f'{PROJECT_DIR}/baseline_results.json', 'r') as f:
    baseline_results = json.load(f)

print(f"Loaded {len(baseline_results)} baseline results")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Setup complete
Project dir: /content/drive/MyDrive/medical-llm
Loaded 20 baseline results


### Cell 3 — Load Fine-Tuned Model (Base Model + LoRA Adapter)

To evaluate our fine-tuned model, we load the original `Mistral-7B-Instruct-v0.2` in 4-bit quantization and then overlay the LoRA adapters stored on the HuggingFace Hub. This approach is highly memory-efficient, requiring only ~5GB of VRAM.

In [ ]:
import gc
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

gc.collect()
torch.cuda.empty_cache()

base_model_name = "mistralai/Mistral-7B-Instruct-v0.2"
adapter_repo = "labdhu/mistral-7b-medical-qa"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_nested_quant=False
)

print("Loading base model...")
tokenizer = AutoTokenizer.from_pretrained(
    base_model_name,
    token=HF_TOKEN,
    trust_remote_code=True
)
tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map={"": 0},
    token=HF_TOKEN,
    trust_remote_code=True,
    low_cpu_mem_usage=True
)

print("Loading LoRA adapter from HuggingFace Hub...")
model = PeftModel.from_pretrained(
    base_model,
    adapter_repo,
    token=HF_TOKEN
)
model.eval()

free_mem, total_mem = torch.cuda.mem_get_info()
print(f"GPU memory: {free_mem/1e9:.1f} GB free / {total_mem/1e9:.1f} GB total")
print("Fine-tuned model loaded successfully")

Loading base model...


config.json:   0%|          | 0.00/596 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/493k [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Loading LoRA adapter from HuggingFace Hub...


adapter_config.json: 0.00B [00:00, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/54.6M [00:00<?, ?B/s]

GPU memory: 2.0 GB free / 15.6 GB total
Fine-tuned model loaded successfully


### Cell 4 — Run Inference on Test Dataset

We iterate through the 20 test questions and generate responses using the fine-tuned model. We use the same instruction prompt format used during training to ensure consistency. The results are stored alongside the baseline responses for direct comparison.

In [ ]:
import json
from tqdm import tqdm

def generate_response(model, tokenizer, instruction, question, max_new_tokens=512):
    prompt = f"<s>[INST] {instruction}\n\n{question} [/INST]"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(
        outputs[0][inputs['input_ids'].shape[1]:],
        skip_special_tokens=True
    )
    return response.strip()

finetuned_results = []

print("Running fine-tuned model on 20 test questions...")
for item in tqdm(baseline_results):
    response = generate_response(
        model, tokenizer,
        item['instruction'],
        item['question']
    )

    finetuned_results.append({
        'question': item['question'],
        'instruction': item['instruction'],
        'ground_truth': item['ground_truth'],
        'baseline_response': item['baseline_response'],
        'finetuned_response': response
    })

with open(f'{PROJECT_DIR}/finetuned_results.json', 'w') as f:
    json.dump(finetuned_results, f, indent=2)

print(f"Saved {len(finetuned_results)} fine-tuned results")

# Show hero examples immediately
hero_indices = [6, 7, 17]  # Q7, Q8, Q18 (0-indexed)
for idx in hero_indices:
    item = finetuned_results[idx]
    print(f"\n{'='*60}")
    print(f"QUESTION: {item['question'][:100]}...")
    print(f"\nBASELINE:\n{item['baseline_response'][:300]}...")
    print(f"\nFINE-TUNED:\n{item['finetuned_response'][:300]}...")

Running fine-tuned model on 20 test questions...


100%|██████████| 20/20 [04:30<00:00, 13.50s/it]

Saved 20 fine-tuned results

QUESTION: My 11 year old son was stung by a bee on his forearm on Monday afternoon. Tuesday morning he woke up...

BASELINE:
I'm an assistant and not a doctor, but I can suggest some steps you can take based on the information provided. It's important to note that I'm not able to provide a definitive diagnosis or medical advice, and you should consult a healthcare professional for this.

It's common for a bee sting to cau...

FINE-TUNED:
hi, bee venom is a neuro-toxin. if the venom is deep in the dermis, we can use benedryl, 25-50 mg, 3 times daily.  there is no need to worry much as the venom is less toxic. no need to worry about the lymph nodes.  if the pain is not reduced in 2 days, he should get an ultrasound done to rule out ly...

QUESTION: I have been taking  Metformim for two weeks doctor has me taken 1 pill at night for 1 to 2 weeks. Si...

BASELINE:
I cannot definitively answer whether you should take two Metformin pills at night without consultin

### Cell 5 — Automated Metrics: ROUGE and BERTScore

We calculate quantitative metrics to measure the improvement. ROUGE measures exact n-gram overlap, while BERTScore provides a semantic similarity score using contextual embeddings. We compare the deltas between the baseline and fine-tuned versions.

In [ ]:
from rouge_score import rouge_scorer
from bert_score import score as bert_score
import pandas as pd
import numpy as np

# ROUGE scoring
scorer = rouge_scorer.RougeScorer(
    ['rouge1', 'rouge2', 'rougeL'],
    use_stemmer=True
)

baseline_rouge = {'rouge1': [], 'rouge2': [], 'rougeL': []}
finetuned_rouge = {'rouge1': [], 'rouge2': [], 'rougeL': []}

for item in finetuned_results:
    ref = item['ground_truth']

    base_scores = scorer.score(ref, item['baseline_response'])
    ft_scores = scorer.score(ref, item['finetuned_response'])

    for metric in ['rouge1', 'rouge2', 'rougeL']:
        baseline_rouge[metric].append(base_scores[metric].fmeasure)
        finetuned_rouge[metric].append(ft_scores[metric].fmeasure)

print("ROUGE Scores:")
print(f"{'Metric':<12} {'Baseline':>10} {'Fine-tuned':>12} {'Delta':>8}")
print("-" * 46)
for metric in ['rouge1', 'rouge2', 'rougeL']:
    base_avg = np.mean(baseline_rouge[metric])
    ft_avg = np.mean(finetuned_rouge[metric])
    delta = ft_avg - base_avg
    print(f"{metric:<12} {base_avg:>10.4f} {ft_avg:>12.4f} {delta:>+8.4f}")

# BERTScore
print("\nComputing BERTScore (this takes 2-3 minutes)...")
references = [item['ground_truth'] for item in finetuned_results]
base_candidates = [item['baseline_response'] for item in finetuned_results]
ft_candidates = [item['finetuned_response'] for item in finetuned_results]

_, _, base_bert_f1 = bert_score(base_candidates, references, lang='en', verbose=False)
_, _, ft_bert_f1 = bert_score(ft_candidates, references, lang='en', verbose=False)

base_bert_avg = base_bert_f1.mean().item()
ft_bert_avg = ft_bert_f1.mean().item()
bert_delta = ft_bert_avg - base_bert_avg

print(f"\nBERTScore F1:")
print(f"{'':12} {'Baseline':>10} {'Fine-tuned':>12} {'Delta':>8}")
print("-" * 46)
print(f"{'BERTScore':<12} {base_bert_avg:>10.4f} {ft_bert_avg:>12.4f} {bert_delta:>+8.4f}")

# Save scores
rouge_results = {
    'baseline_rouge': {k: float(np.mean(v)) for k, v in baseline_rouge.items()},
    'finetuned_rouge': {k: float(np.mean(v)) for k, v in finetuned_rouge.items()},
    'baseline_bertscore': base_bert_avg,
    'finetuned_bertscore': ft_bert_avg
}

with open(f'{PROJECT_DIR}/rouge_bertscore_results.json', 'w') as f:
    json.dump(rouge_results, f, indent=2)

print("\nRouge and BERTScore results saved.")

ROUGE Scores:
Metric         Baseline   Fine-tuned    Delta
----------------------------------------------
rouge1           0.2433       0.3788  +0.1355
rouge2           0.0749       0.1473  +0.0724
rougeL           0.1442       0.2464  +0.1022

Computing BERTScore (this takes 2-3 minutes)...


config.json:   0%|          | 0.00/482 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
pooler.dense.weight             | MISSING    | 
pooler.dense.bias               | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.



BERTScore F1:
               Baseline   Fine-tuned    Delta
----------------------------------------------
BERTScore        0.8436       0.8696  +0.0260

Rouge and BERTScore results saved.


### Cell 6 — LLM-as-Judge Evaluation (Llama-3.3-70B via Groq)

We use Llama-3.3-70B served via Groq as an impartial judge to grade responses on medical accuracy, professional tone, and conciseness. Using a strong open-source model as judge demonstrates that LLM-as-Judge evaluation is model-agnostic — the methodology works regardless of which judge model is used. Groq provides free API access with generous rate limits, making this approach reproducible without cost barriers.

In [8]:
!pip install -q groq

import json
import time
import numpy as np
from groq import Groq
from google.colab import userdata

groq_client = Groq(api_key=userdata.get('GROQ_API_KEY'))

def judge_response(question, response, ground_truth):
    prompt = f"""You are a medical AI evaluation expert. Grade the following response to a medical question.

Question: {question}

Ground Truth Answer: {ground_truth}

Response to Grade: {response}

Grade this response on exactly these three criteria. Respond ONLY with a JSON object, no other text:

{{
  "medical_accuracy": <integer 1-5>,
  "professional_tone": <integer 1-5>,
  "conciseness": <integer 1-5>,
  "reasoning": "<one sentence explanation>"
}}

Scoring guide:
- medical_accuracy: 1=factually wrong, 3=partially correct, 5=accurate and complete
- professional_tone: 1=full of disclaimers/deflection, 3=neutral, 5=responds like a medical professional
- conciseness: 1=rambling/repetitive, 3=acceptable length, 5=direct and structured
"""
    try:
        response_obj = groq_client.chat.completions.create(
            model="llama-3.3-70b-versatile",
            messages=[{"role": "user", "content": prompt}],
            temperature=0
        )
        text = response_obj.choices[0].message.content.strip()
        if text.startswith('```'):
            text = text.split('```')[1]
            if text.startswith('json'):
                text = text[4:]
        return json.loads(text.strip())
    except Exception as e:
        print(f"Judge error: {e}")
        return {"medical_accuracy": 0, "professional_tone": 0, "conciseness": 0, "reasoning": "error"}

with open(f'{PROJECT_DIR}/finetuned_results.json', 'r') as f:
    finetuned_results = json.load(f)

judge_results = []

print("Running LLM-as-Judge with Llama-3.3-70B via Groq...")
for i, item in enumerate(finetuned_results):
    print(f"Grading question {i+1}/20...")

    base_grade = judge_response(
        item['question'],
        item['baseline_response'],
        item['ground_truth']
    )
    time.sleep(2)

    ft_grade = judge_response(
        item['question'],
        item['finetuned_response'],
        item['ground_truth']
    )
    time.sleep(2)

    judge_results.append({
        'question': item['question'][:100],
        'baseline_grades': base_grade,
        'finetuned_grades': ft_grade
    })
    print(f"  Baseline: {base_grade}")
    print(f"  Fine-tuned: {ft_grade}")

criteria = ['medical_accuracy', 'professional_tone', 'conciseness']
print("\nLLM-as-Judge Results:")
print(f"{'Criteria':<22} {'Baseline':>10} {'Fine-tuned':>12} {'Delta':>8}")
print("-" * 56)

for criterion in criteria:
    base_scores = [r['baseline_grades'].get(criterion, 0) for r in judge_results]
    ft_scores = [r['finetuned_grades'].get(criterion, 0) for r in judge_results]
    base_avg = np.mean(base_scores)
    ft_avg = np.mean(ft_scores)
    delta = ft_avg - base_avg
    print(f"{criterion:<22} {base_avg:>10.2f} {ft_avg:>12.2f} {delta:>+8.2f}")

with open(f'{PROJECT_DIR}/llm_judge_results.json', 'w') as f:
    json.dump(judge_results, f, indent=2)

print("\nLLM-as-Judge results saved successfully.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.3/142.3 kB 7.0 MB/s eta 0:00:00
Running LLM-as-Judge with Llama-3.3-70B via Groq...
Grading question 1/20...
  Baseline: {'medical_accuracy': 4, 'professional_tone': 5, 'conciseness': 4, 'reasoning': 'The response is mostly accurate, well-structured, and written in a professional tone, but lacks some details and specificity found in the ground truth answer.'}
  Fine-tuned: {'medical_accuracy': 4, 'professional_tone': 5, 'conciseness': 4, 'reasoning': 'The response is mostly accurate, well-structured, and professional, but lacks some details and specificity compared to the ground truth answer.'}
Grading question 2/20...
  Baseline: {'medical_accuracy': 5, 'professional_tone': 5, 'conciseness': 4, 'reasoning': 'The response accurately identifies the possibility of a tooth infection, provides relevant advice, and maintains a professional tone, although it could be slightly more concise.'}
  Fine-tuned: {'medical_accuracy': 3, 'professional_t

### Cell 7 — Visual Comparison & Summary

We present the final side-by-side comparison of the three 'hero' examples that previously failed the baseline test. This section summarizes the project achievements, highlighting the validation loss improvement and the qualitative shift from evasive answers to structured medical guidance.

In [9]:
import textwrap

def print_comparison(item, label):
    print(f"\n{'='*70}")
    print(f"HERO EXAMPLE — {label}")
    print(f"{'='*70}")
    print(f"QUESTION: {item['question'][:150]}")
    print(f"\n--- BASELINE RESPONSE ---")
    print(textwrap.fill(item['baseline_response'][:500], width=70))
    print(f"\n--- FINE-TUNED RESPONSE ---")
    print(textwrap.fill(item['finetuned_response'][:500], width=70))

labels = [
    "Q7 — Disclaimer Wall (Bee Sting)",
    "Q8 — Evasive Non-Answer (Metformin)",
    "Q18 — Symptom Echo + Cutoff (Pelvic Pain)"
]

hero_indices = [6, 7, 17]

print("BEFORE vs AFTER — THREE HERO EXAMPLES")
for idx, label in zip(hero_indices, labels):
    print_comparison(finetuned_results[idx], label)

print("\n\nEVALUATION SUMMARY")
print("="*70)
print(f"Training: 8,000 medical Q&A pairs, 1 epoch, QLoRA r=16")
print(f"Validation loss improvement: 1.877 -> 1.846")
print(f"Token accuracy: 58.5%")

print("\nSee rouge_bertscore_results.json and llm_judge_results.json for full scores.")

BEFORE vs AFTER — THREE HERO EXAMPLES

HERO EXAMPLE — Q7 — Disclaimer Wall (Bee Sting)
QUESTION: My 11 year old son was stung by a bee on his forearm on Monday afternoon. Tuesday morning he woke up in extreme pain and complained of his armpit hurt

--- BASELINE RESPONSE ---
I'm an assistant and not a doctor, but I can suggest some steps you
can take based on the information provided. It's important to note
that I'm not able to provide a definitive diagnosis or medical advice,
and you should consult a healthcare professional for this.  It's
common for a bee sting to cause localized pain, redness, and swelling.
However, your son's symptoms of severe pain, extreme redness, and
swelling spreading to the armpit, along with swollen lymph nodes,
could indicate a more seri

--- FINE-TUNED RESPONSE ---
hi, bee venom is a neuro-toxin. if the venom is deep in the dermis, we
can use benedryl, 25-50 mg, 3 times daily.  there is no need to worry
much as the venom is less toxic. no need to worry abou